<a href="https://colab.research.google.com/github/elkins/synth-saxs/blob/main/examples/interactive_tutorials/saxs_profile_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAXS Profile Generation & Analysis

This tutorial demonstrates how to compute Small-Angle X-ray Scattering (SAXS) profiles directly from atomic coordinates using the Debye formula, and how to analyze the resulting data (Radius of Gyration, Kratky plots).

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q synth-saxs biotite matplotlib
else:
    sys.path.append("../../")

import biotite.structure.io as strucio
import matplotlib.pyplot as plt

## 1. Load a Structure
We will download a PDB file (e.g., Lysozyme, 1AKI) using Biotite.

In [ ]:
import biotite.database.rcsb as rcsb

# Download and load Lysozyme
pdb_file = rcsb.fetch("1AKI", "pdb", ".")
structure = strucio.load_structure(pdb_file)
# Filter out water and hetero atoms
protein = structure[~structure.hetero]

# Extract coordinates and elements
coords = protein.coord
elements = protein.element

print(f"Loaded protein with {len(coords)} atoms.")

## 2. Simulate the SAXS Profile
The Debye scattering formula relates the 3D distances between all pairs of atoms to the 1D scattering intensity $I(q)$:
$$ I(q) = \sum_{i,j} f_i(q) f_j(q) rac{\sin(q r_{ij})}{q r_{ij}} $$

In [ ]:
from synth_saxs import add_noise, calculate_saxs_profile

# Calculate SAXS profile
q_values, intensities = calculate_saxs_profile(protein, q_min=0.01, q_max=0.5, n_points=100)

# Add synthetic experimental noise
noisy_intensities = add_noise(intensities, noise_level=0.05)

## 3. Visualization
Let's plot the standard $I(q)$ vs $q$ curve and a Kratky plot ($I(q) \cdot q^2$ vs $q$) to assess foldedness.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Standard SAXS profile (Log scale)
ax1.plot(q_values, noisy_intensities, "ko", markersize=4, alpha=0.5, label="Simulated Data")
ax1.plot(q_values, intensities, "r-", linewidth=2, label="Ideal Profile")
ax1.set_yscale("log")
ax1.set_xlabel("q (1/Å)", fontsize=12)
ax1.set_ylabel("Intensity I(q)", fontsize=12)
ax1.set_title("Simulated SAXS Profile", fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Kratky Plot
# Folded proteins show a bell-shaped peak, unfolded proteins plateau or increase
kratky_y = intensities * (q_values**2)
ax2.plot(q_values, kratky_y, "b-", linewidth=2)
ax2.set_xlabel("q (1/Å)", fontsize=12)
ax2.set_ylabel("I(q) * q²", fontsize=12)
ax2.set_title("Kratky Plot", fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Compute Radius of Gyration ($R_g$)
We can analytically compute the $R_g$ directly from coordinates, which corresponds to the Guinier approximation at low $q$ angles.

In [ ]:
from synth_saxs import calculate_radius_of_gyration

rg = calculate_radius_of_gyration(protein)
print(f"Analytic Radius of Gyration (Rg): {rg:.2f} Å")